# Does Newton get the stretch right? — the hanging bar

**The question.** NVIDIA Newton's **XPBD** solver is built for speed (games, robotics). How much physical accuracy does that speed cost?

**The test.** We clamp a soft bar at the top and let gravity stretch it — pure tension under self-weight. Here the analytic answer is the **deformation itself** (a textbook self-weight bar), so every solver is scored against it node-for-node, not just against each other — the contact / friction / material tests carry analytic anchors too, but for a force or stress.

**The solvers compared** — same bar, same mesh, same gravity; *only the solver differs*:

| | what it is |
|---|---|
| **Newton XPBD** | *Extended Position-Based Dynamics* — fast positional projection (the game/robotics solver); leaves a finite force-balance residual |
| **Newton VBD** | *Vertex Block Descent* — implicit; minimises the backward-Euler objective; slow to converge on this slender bar |
| **Newton SemiImplicit** | explicit, force-based (symplectic Euler); Newton's *differentiable* solver |
| **FEM (FEniCSx)** | reference implicit finite elements |
| **analytic bar** | the closed-form textbook answer |

**The result is in the next chart:** all three Newton solvers settle markedly soft at this run's budgets — XPBD ≈ 3.7×, VBD ≈ 3.2×, explicit ≈ 2.0× the FEM tip — while FEM lands within a few percent of the analytic bar. §3 shows *why* XPBD never reaches a force balance; VBD is soft for a different reason (its block Gauss-Seidel has not converged on this slender bar at the iteration budget used).

In [ ]:
%matplotlib inline
import os

import matplotlib.pyplot as plt
import numpy as np

from common import params
from compare import energies as en
from compare import hanging_bar as cmp_hb
from compare import style


def _opt(path):
    return np.load(path) if os.path.exists(path) else None

newton = np.load(params.NEWTON_NPZ)          # Newton XPBD (positional, canonical run)
vbd    = _opt(params.NEWTON_VBD_NPZ)         # Newton VBD (implicit)
semi   = _opt(params.NEWTON_SEMI_NPZ)        # Newton explicit (SemiImplicit, differentiable)
fem    = np.load(params.FEM_NPZ)             # FEM tet (same mesh, node-for-node)
hexfem = _opt(params.FEM_HEX_NPZ)            # FEM hex (independent mesh)
tets   = newton["tet_indices"]

# (label, data, colour) for whichever Newton solvers are present on disk
newton_runs = [(style.LABEL["xpbd"], newton, style.COLOR["xpbd"])]
if vbd is not None:
    newton_runs.append((style.LABEL["vbd"], vbd, style.COLOR["vbd"]))
if semi is not None:
    newton_runs.append((style.LABEL["semi_implicit"], semi, style.COLOR["semi_implicit"]))

print(params.summary())
print("Newton solvers present:", [n for n, _, _ in newton_runs])

## The scene — what we're actually simulating

A soft bar (0.6 × 0.6 × 1.6 m) is clamped on its **top** face and hangs under its own weight. Below is Newton's settled result, rendered straight from the simulation mesh: colour = how far each point dropped (dark at the clamp, bright at the free end), with the undeformed shape ghosted behind it. This is the whole experiment — everything after just quantifies it.

In [ ]:
from compare import scene

fig = plt.figure(figsize=(5, 6))
ax = fig.add_subplot(111, projection="3d")
norm, lab = scene.render(ax, newton["rest_q"], newton["final_q"], tets,
                         title="Hanging bar - clamped on top, stretched by gravity")
scene.add_colorbar(fig, ax, norm, lab)
ax.view_init(elev=12, azim=-70)
plt.tight_layout(); plt.show()

## 1. The verdict — how far does the tip drop?

The single number that matters: how far the free end of the bar drops under its own weight. The **analytic bar** is the closed-form 1-D reference (a sanity anchor — it omits Poisson/3-D effects, so not the exact 3-D answer), **FEM** is the trusted numerical reference, and the **Newton** solvers are what we are testing. Read the bars left to right — Newton solvers, then FEM (tet / hex), then the analytic 1-D reference.

In [ ]:
def tip_drop(d):
    free = np.setdiff1d(np.arange(len(d["rest_q"])), d["fixed_nodes"])
    return -(d["final_q"] - d["rest_q"])[free, 2].min() * 1000.0

z_top = newton["rest_q"][newton["fixed_nodes"], 2].mean()
vals, colors = {}, []
for label, d, c in newton_runs:
    vals[label] = tip_drop(d); colors.append(c)
vals["FEM tet"] = tip_drop(fem); colors.append(style.COLOR["fem"])
if hexfem is not None:
    vals["FEM hex"] = tip_drop(hexfem); colors.append(style.COLOR["fem_hex"])
vals["analytic 1-D"] = params.analytic_hanging_displacement(
    newton["rest_q"][:, 2].min(), z_top, params.BLOCK_LZ) * 1000.0
colors.append(style.COLOR["analytic"])
for k, v in vals.items():
    print(f"{k:16s}: {v:7.2f} mm")
print(f"\nVerdict: XPBD tip {vals['Newton XPBD']:.1f} mm  vs  FEM tet {vals['FEM tet']:.1f} mm  "
      f"vs  analytic {vals['analytic 1-D']:.1f} mm   (XPBD / FEM = {vals['Newton XPBD'] / vals['FEM tet']:.2f})")
plt.figure(figsize=(7, 4))
plt.bar(list(vals), list(vals.values()), color=colors)
plt.ylabel("tip drop [mm]"); plt.title("Hanging bar - tip deflection (the verdict)")
plt.xticks(rotation=15); plt.tight_layout(); plt.show()

## 2. Can we trust the FEM reference?

A skeptic's first question: *why believe FEM at all?* Because it lands on the closed-form analytic bar. The analytic 1-D self-weight bar (`u_tip = ρ g L² / 2E`) ignores Poisson contraction (here ν = 0.42) and 3-D effects, so we expect FEM *close*, not identical. Linear tets **lock** (come out too stiff); Hex8 locks less and sits closer to analytic — it is the more trustworthy element. FEM landing within a few percent of the textbook answer is what makes it a credible reference; the mesh-refinement limit is quantified in `30_convergence.ipynb`.

## 3. Why XPBD reads soft — the force balance it never reaches

This is the mechanism, and the most honest single measure of *solve vs. project*. At static equilibrium the internal elastic force must exactly cancel gravity at every free node (`f = −∂U/∂x + f_grav ≈ 0`). **FEM solves that equation**, so its residual is ~0. **XPBD projects positions** to satisfy constraints but never enforces the force balance, so it leaves a finite out-of-balance force — and that residual *is* how far its settled state sits from the true equilibrium. **VBD** is implicit, so with enough iterations it *would* drive the residual down — but at the budget used here it has not converged on this slender bar, so it too settles soft (≈ 3.2× the FEM tip) with a residual still well above zero. (The force model used here is unit-tested in `tests/test_energies.py`.)

In [ ]:
def resid(d):
    return en.equilibrium_residual(d["rest_q"], d["final_q"], tets, d["fixed_nodes"])

rf = resid(fem)
plt.figure(figsize=(6, 4))
for label, d, c in newton_runs:
    r = resid(d)
    print(f"{label:16s}: free-node residual RMS = {r['free_rms']:.4g} N | "
          f"reaction_z = {r['reaction_z']:.4g} N (weight = {r['weight']:.4g} N)")
    plt.hist(np.linalg.norm(r["residual"][r["free"]], axis=1), bins=40, alpha=0.6, label=label, color=c)
print(f"{'FEM tet':16s}: free-node residual RMS = {rf['free_rms']:.4g} N | "
      f"reaction_z = {rf['reaction_z']:.4g} N (weight = {rf['weight']:.4g} N)")
plt.hist(np.linalg.norm(rf["residual"][rf["free"]], axis=1), bins=40, alpha=0.6, label="FEM tet", color=style.COLOR["fem"])
plt.xlabel("per-node out-of-balance force [N]"); plt.ylabel("count"); plt.legend()
plt.title("Hanging bar - equilibrium residual (FEM ~ 0, XPBD finite)"); plt.show()

## 4. Same shape along the bar?

Not just the tip — the whole stretch profile (downward displacement vs. original height). If a solver matches FEM and the analytic bar along the *entire* length, the agreement is structural, not a lucky single number.

In [ ]:
# Single-source figure: the same make_profile the pipeline saves to figures/
# (compare/hanging_bar.py) -- fix it once, both the notebook and the pipeline update.
cmp_hb.make_profile(newton_runs, fem, hexfem)
plt.show()

## 5. Same stored energy?

The released gravitational potential energy is stored as elastic strain energy. It is evaluated with the **same** Neo-Hookean density on the **same** mesh for every solver (`compare/energies.py`, unit-tested), so the numbers are directly comparable — a softer-looking solver stores more strain energy for the same load.

In [ ]:
m = en.node_masses(newton["rest_q"], tets)
U, ucolors = {}, []
for label, d, c in newton_runs:
    U[label] = en.strain_energy(d["rest_q"], d["final_q"], tets); ucolors.append(c)
U["FEM tet"] = en.strain_energy(fem["rest_q"], fem["final_q"], tets); ucolors.append(style.COLOR["fem"])
U["analytic"] = en.analytic_hanging_strain_energy(); ucolors.append(style.COLOR["analytic"])
for k in U:
    print(f"strain energy  {k:16s}: {U[k]:.4g} J")
plt.figure(figsize=(7, 4))
plt.bar(list(U), list(U.values()), color=ucolors)
plt.ylabel("strain energy [J]"); plt.title("Hanging bar - internal energy")
plt.tight_layout(); plt.show()

## 6. What does the speed buy? — computation time

The whole point of XPBD is speed; here is the trade-off, measured. Solver wall-clock time (the solve only, not mesh build or IO). Newton runs on the GPU; the implicit FEM solve is the accurate-but-heavier reference.

In [ ]:
times, tcolors = {}, []
for label, d, c in newton_runs:
    if "wall_time" in d.files:
        times[label] = float(d["wall_time"]); tcolors.append(c)
times["FEM tet"] = float(fem["wall_time"]); tcolors.append(style.COLOR["fem"])
if hexfem is not None and "wall_time" in hexfem.files:
    times["FEM hex"] = float(hexfem["wall_time"]); tcolors.append(style.COLOR["fem_hex"])
for k, v in times.items():
    print(f"{k:16s}: {v:8.3f} s")
plt.figure(figsize=(7, 4))
plt.bar(list(times), list(times.values()), color=tcolors)
plt.ylabel("solve wall time [s]"); plt.title("Hanging bar - computation time")
plt.xticks(rotation=15); plt.tight_layout(); plt.show()

## 7. Volume handling — the Jacobian (det F)

How each solver distributes volumetric strain. J = det F = 1 means volume preserved; the spread of J across elements shows how the volumetric response differs across the fast projection (XPBD), the implicit solvers (VBD / explicit) and FEM.

In [ ]:
# Volume handling for ALL three Newton solvers vs FEM (J = det F per tet).
Jf = en.jacobians(fem["rest_q"], fem["final_q"], tets)
dVf = en.volume_change(fem["rest_q"], fem["final_q"], tets) * 100
print(f"{'solver':16s}  J  mean / min / max          dV")
for label, d, c in newton_runs:
    J = en.jacobians(d["rest_q"], d["final_q"], tets)
    dV = en.volume_change(d["rest_q"], d["final_q"], tets) * 100
    print(f"{label:16s}: {J.mean():.4f} / {J.min():.4f} / {J.max():.4f}    {dV:+.2f}%")
print(f"{'FEM tet':16s}: {Jf.mean():.4f} / {Jf.min():.4f} / {Jf.max():.4f}    {dVf:+.2f}%")

plt.figure(figsize=(6, 4))
for label, d, c in newton_runs:
    plt.hist(en.jacobians(d["rest_q"], d["final_q"], tets), bins=40, alpha=0.5, label=label, color=c)
plt.hist(Jf, bins=40, alpha=0.5, label="FEM tet", color=style.COLOR["fem"])
plt.axvline(1.0, color="k", ls="--", lw=1)
plt.xlabel("per-tet J = det F"); plt.ylabel("count"); plt.legend()
plt.title("Hanging bar - Jacobian distribution (all solvers vs FEM)"); plt.show()

## 8. Settling — did the dynamic solvers reach rest, and at the right place?

Newton is a *dynamic* solver driven to a static answer: gravity is applied suddenly, the bar rings, and per-frame velocity damping drains the kinetic energy until it settles. **Right panel:** the sanity check that each Newton solver truly reached rest (KE → 0) before we read its "settled" state. **Left panel:** the tip-height trajectory against the static **FEM** and **analytic** targets (horizontal lines — FEM has no transient of its own), so we see not only *whether* each solver comes to rest but *where* it rests relative to the reference.

In [ ]:
# Single-source figure: the same make_settling the pipeline saves to figures/.
cmp_hb.make_settling(newton_runs, fem)
plt.show()

## 9. A differentiable stiffness fit (θ\*)

Newton is built on differentiable Warp, so we can backpropagate through the whole simulation. We fit a stiffness multiplier θ so the (differentiable) **SemiImplicit** solver matches FEM. The fitted **θ\*** is the *effective-stiffness multiplier* of that Newton model vs FEM (θ\* > 1 → effectively softer). Note this characterises the SemiImplicit solver, **not** XPBD — XPBD's softness is the residual in §3.

Run `python -m newton_run.diffsim` (CUDA) to produce the data.

In [ ]:
if os.path.exists(params.DIFFSIM_NPZ):
    ds = np.load(params.DIFFSIM_NPZ)
    print(f"Sensitivity at theta=1 : loss = {float(ds['loss0']):.4e} m^2, dLoss/dtheta = {float(ds['grad0']):.4e}")
    print(f"Fitted multiplier theta* = {float(ds['theta_star']):.4f}  "
          f"({'Newton model softer than FEM' if float(ds['theta_star']) > 1 else 'Newton model stiffer than FEM'})")
    fig, ax1 = plt.subplots(figsize=(6, 4))
    ax1.semilogy(ds["loss_history"], color=style.NEUTRAL[0]); ax1.set_xlabel("iteration")
    ax1.set_ylabel("mismatch loss [m^2]", color=style.NEUTRAL[0])
    ax2 = ax1.twinx(); ax2.plot(ds["theta_history"], color=style.NEUTRAL[1])
    ax2.set_ylabel("theta (stiffness mult.)", color=style.NEUTRAL[1]); ax2.axhline(1.0, color="grey", ls=":", lw=1)
    plt.title("Hanging bar - differentiable stiffness fit (SemiImplicit)"); plt.tight_layout(); plt.show()
else:
    print("No diffsim result yet. Run: python -m newton_run.diffsim")
    print("(Newton differentiates through the SemiImplicit sim to fit the effective stiffness vs FEM.)")

## Takeaway (the 10-minute summary)

* On this test, whose **deformation has a closed form**, FEM lands within a few percent of the analytic bar — but all three Newton solvers settle markedly softer at this run's budgets (XPBD ≈ 3.7×, VBD ≈ 3.2×, explicit ≈ 2.0× the FEM tip); the **explicit** solver is the closest of the three and VBD barely improves on XPBD (§1).
* The reasons differ by solver: **XPBD never reaches a force balance** — it leaves a finite equilibrium residual (§3) — because it *projects* positions instead of *solving* R(u) = 0 (mechanical, not a tuning artefact). **VBD** *is* implicit and would approach FEM with enough iterations, but at the budget used its block Gauss-Seidel has not converged on this 17-layer bar, so it stays ~3.2× soft — a convergence-budget effect, not a force-balance one.
* That is the trade you are buying: **XPBD is fast on the GPU (§6)** at the cost of an unforced equilibrium; **VBD** is implicit but, at the fast budget here, still settles far soft (closing the gap costs many more iterations — see `30_convergence`); **FEM** is the force-balanced reference. Every diagnostic is computed identically for both sides; the underlying force/energy model is **unit-tested** (`pytest tests/`).